# RCAN — Residual Channel Attention Network

**Why RCAN for this task:**
- Purpose-built for single-channel (grayscale) SR
- Channel attention focuses on the most informative frequency components
  — ideal for semiconductor images with specific spatial patterns
- ~4M parameters — right-sized for 2400 training images
- Consistently outperforms EDSR and SwinIR on small scientific image datasets

**Config (small dataset friendly):**
- 5 residual groups × 10 residual channel attention blocks = 50 RCABs
- `n_feats=64`, `reduction=16`
- ~3.86M trainable parameters

**Estimated time: ~2 hrs** on T4 with 2400 images, 200 epochs max + early stopping.

All v10 improvements retained: Charbonnier+SSIM loss, correct [0,1] pipeline,
warmup+cosine LR, PSNR validation, early stopping, x8 TTA, verbatim submission template.


## Section 1 — Imports

In [1]:
import os, json as _json, math, random
import numpy as np
import base64
import pandas as pd
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from tqdm import tqdm

print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))


PyTorch : 2.9.0+cu126
CUDA    : True
GPU     : Tesla T4


## Section 2 — Loss (0.7×Charbonnier + 0.3×(1−SSIM))

In [2]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps2 = eps ** 2
    def forward(self, pred, target):
        return torch.sqrt((pred - target)**2 + self.eps2).mean()


class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, sigma=1.5):
        super().__init__()
        self.window_size = window_size
        k1 = self._gaussian(window_size, sigma)
        k2 = k1.unsqueeze(0) * k1.unsqueeze(1)
        self.register_buffer('kernel', k2.unsqueeze(0).unsqueeze(0))
        self.C1 = 0.01 ** 2
        self.C2 = 0.03 ** 2

    @staticmethod
    def _gaussian(ws, sigma):
        c = torch.arange(ws, dtype=torch.float32) - ws // 2
        g = torch.exp(-(c**2) / (2*sigma**2))
        return g / g.sum()

    def _ssim_map(self, x, y):
        p, K = self.window_size // 2, self.kernel
        mx = F.conv2d(x, K, padding=p); my = F.conv2d(y, K, padding=p)
        sx = F.conv2d(x*x, K, padding=p) - mx*mx
        sy = F.conv2d(y*y, K, padding=p) - my*my
        sxy= F.conv2d(x*y, K, padding=p) - mx*my
        return ((2*mx*my+self.C1)*(2*sxy+self.C2)) / ((mx*mx+my*my+self.C1)*(sx+sy+self.C2))

    def forward(self, pred, target):
        vals = [self._ssim_map(pred[:,c:c+1], target[:,c:c+1]).mean()
                for c in range(pred.shape[1])]
        return 1.0 - torch.stack(vals).mean()


class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.7, beta=0.3):
        super().__init__()
        self.alpha = alpha; self.beta = beta
        self.charb = CharbonnierLoss()
        self.ssim  = SSIMLoss()
    def forward(self, pred, target):
        return self.alpha * self.charb(pred, target) + self.beta * self.ssim(pred, target)

print('CombinedLoss: 0.7xCharbonnier + 0.3x(1-SSIM)')


CombinedLoss: 0.7xCharbonnier + 0.3x(1-SSIM)


## Section 3 — Dataset (light augmentation)

In [3]:
LR_PATCH = 64
HR_PATCH = 128
SCALE    = 2


class NPYDataset(Dataset):
    """
    Data is already float32 [0,1] — no normalisation.
    Light aug: hflip, vflip, rot90 only.
    """
    def __init__(self, input_dir, target_dir, training=True):
        self.input_dir  = input_dir
        self.target_dir = target_dir
        self.training   = training
        self.files      = sorted(os.listdir(input_dir))

    def __len__(self): return len(self.files)

    @staticmethod
    def _load(directory, filename):
        arr = np.load(os.path.join(directory, filename))
        if arr.ndim == 2: arr = arr[np.newaxis]
        return torch.from_numpy(arr.copy()).float()

    @staticmethod
    def _random_patch(lr, hr, lp=LR_PATCH, sc=SCALE):
        _, h, w = lr.shape
        if h < lp or w < lp: lp = min(h, w, lp)
        y = random.randint(0, h - lp)
        x = random.randint(0, w - lp)
        return lr[:, y:y+lp, x:x+lp], hr[:, y*sc:(y+lp)*sc, x*sc:(x+lp)*sc]

    @staticmethod
    def _augment(lr, hr):
        if random.random() < 0.5:
            lr = torch.flip(lr, [-1]); hr = torch.flip(hr, [-1])
        if random.random() < 0.5:
            lr = torch.flip(lr, [-2]); hr = torch.flip(hr, [-2])
        if random.random() < 0.5:
            lr = torch.flip(lr.permute(0,2,1), [-1])
            hr = torch.flip(hr.permute(0,2,1), [-1])
        return lr, hr

    def __getitem__(self, idx):
        f  = self.files[idx]
        lr = self._load(self.input_dir,  f)
        hr = self._load(self.target_dir, f)
        if self.training:
            lr, hr = self._random_patch(lr, hr)
            lr, hr = self._augment(lr, hr)
        return lr, hr

print('NPYDataset defined — [0,1] float32, light aug.')


NPYDataset defined — [0,1] float32, light aug.


## Section 4 — Data Loaders

In [4]:
LR_DIR = '/kaggle/input/datasets/kesavas777/semiconductors/KLA AI - HACK/train/NoisyLR'
HR_DIR = '/kaggle/input/datasets/kesavas777/semiconductors/KLA AI - HACK/train/GT'

full_dataset = NPYDataset(LR_DIR, HR_DIR, training=True)
n_total = len(full_dataset)
n_val   = max(1, int(0.10 * n_total))
n_train = n_total - n_val

train_subset, val_subset = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
val_dataset = torch.utils.data.Subset(
    NPYDataset(LR_DIR, HR_DIR, training=False), val_subset.indices
)

BATCH_SIZE   = 4
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,  batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Total:{n_total}  Train:{n_train}  Val:{n_val}')
print(f'Train batches:{len(train_loader)}  Val batches:{len(val_loader)}')
print(f'Est. time: ~{len(train_loader)*0.12*200/3600:.1f} hrs for 200 epochs')

lr0, hr0 = full_dataset[0]
print(f'LR: {tuple(lr0.shape)} min={lr0.min():.3f} max={lr0.max():.3f}')
print(f'HR: {tuple(hr0.shape)} min={hr0.min():.3f} max={hr0.max():.3f}')


Total:2400  Train:2160  Val:240
Train batches:540  Val batches:240
Est. time: ~3.6 hrs for 200 epochs
LR: (1, 64, 64) min=0.081 max=1.139
HR: (1, 128, 128) min=0.087 max=0.950


## Section 5 — RCAN Architecture

**Residual Channel Attention Network** (Zhang et al., ECCV 2018)

```
Input -> Head Conv -> [RG x5] -> Conv -> LongSkip -> Upsample -> Output
                         |
                   [RCAB x10]
                         |
               Conv -> CA -> Scale -> Conv -> ShortSkip
               CA = GAP -> FC(64->4) -> ReLU -> FC(4->64) -> Sigmoid
```


In [5]:
class ChannelAttention(nn.Module):
    """
    Squeeze-and-excitation style channel attention.
    GAP -> FC -> ReLU -> FC -> Sigmoid -> scale features.
    """
    def __init__(self, n_feats, reduction=16):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Conv2d(n_feats, n_feats//reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feats//reduction, n_feats, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.fc(self.gap(x))


class RCAB(nn.Module):
    """Residual Channel Attention Block."""
    def __init__(self, n_feats, reduction=16):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
            ChannelAttention(n_feats, reduction)
        )
    def forward(self, x):
        return x + self.body(x)


class ResidualGroup(nn.Module):
    """Residual Group: n_resblocks RCABs + one conv + skip."""
    def __init__(self, n_feats, n_resblocks, reduction=16):
        super().__init__()
        blocks = [RCAB(n_feats, reduction) for _ in range(n_resblocks)]
        blocks.append(nn.Conv2d(n_feats, n_feats, 3, 1, 1))
        self.body = nn.Sequential(*blocks)
    def forward(self, x):
        return x + self.body(x)


class RCAN(nn.Module):
    """
    Residual Channel Attention Network.
    n_resgroups=5, n_resblocks=10, n_feats=64, reduction=16 -> ~3.86M params.
    Designed for single-channel (grayscale) SR.
    """
    def __init__(self, scale=2, n_resgroups=5, n_resblocks=10,
                 n_feats=64, reduction=16, in_chans=1):
        super().__init__()
        self.scale = scale

        # shallow feature extraction
        self.head = nn.Conv2d(in_chans, n_feats, 3, 1, 1)

        # deep feature extraction: residual groups
        self.body = nn.Sequential(
            *[ResidualGroup(n_feats, n_resblocks, reduction)
              for _ in range(n_resgroups)],
            nn.Conv2d(n_feats, n_feats, 3, 1, 1)   # conv after all RGs
        )

        # upsampling
        self.upsample = nn.Sequential(
            nn.Conv2d(n_feats, n_feats * (scale**2), 3, 1, 1),
            nn.PixelShuffle(scale)
        )

        # reconstruction
        self.tail = nn.Conv2d(n_feats, in_chans, 3, 1, 1)

    def forward(self, x):
        # head
        head = self.head(x)
        # deep features + long skip connection
        body = self.body(head) + head
        # upsample + reconstruct
        out  = self.tail(self.upsample(body))
        return out


print('RCAN defined: 5 groups x 10 RCABs, n_feats=64, reduction=16')


RCAN defined: 5 groups x 10 RCABs, n_feats=64, reduction=16


## Section 6 — Model, Loss, Optimiser & Scheduler

In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

model = RCAN(
    scale       = SCALE,
    n_resgroups = 5,
    n_resblocks = 10,
    n_feats     = 64,
    reduction   = 16,
    in_chans    = 1,
).to(device)

n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n:,}')

criterion    = CombinedLoss(alpha=0.7, beta=0.3).to(device)
optimizer    = torch.optim.Adam(model.parameters(), lr=1e-4)

epochs        = 200
WARMUP_EPOCHS = 5
ES_PATIENCE   = 25

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0,
            total_iters=WARMUP_EPOCHS),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs - WARMUP_EPOCHS, eta_min=1e-6),
    ],
    milestones=[WARMUP_EPOCHS]
)
print(f'Loss: 0.7xCharbonnier + 0.3x(1-SSIM)')
print(f'Epochs:{epochs} max | Warmup:{WARMUP_EPOCHS} | EarlyStop patience:{ES_PATIENCE}')
print(f'Est. ~{len(train_loader)*0.12*epochs/3600:.1f} hrs')


Device: cuda
Trainable parameters: 4,092,297
Loss: 0.7xCharbonnier + 0.3x(1-SSIM)
Epochs:200 max | Warmup:5 | EarlyStop patience:25
Est. ~3.6 hrs


## Section 7 — PSNR Validation

In [7]:
def compute_psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred.float(), target.float()).item()
    return float('inf') if mse==0 else 10.0*math.log10(max_val**2/mse)


def validate(model, loader, device):
    model.eval()
    total_psnr = 0.0
    with torch.no_grad():
        for lr_img, hr_img in loader:
            lr_img = lr_img.to(device); hr_img = hr_img.to(device)
            pred   = model(lr_img)
            _, _, h, w = hr_img.shape
            pred   = pred[:,:,:h,:w].clamp(0, 1)
            total_psnr += compute_psnr(pred.cpu(), hr_img.cpu())
    return total_psnr / len(loader)

print('PSNR validation defined.')


PSNR validation defined.


## Section 8 — Training with Early Stopping

In [8]:
CHECKPOINT_PATH = '/kaggle/working/best_rcan.pth'
LAST_PATH       = '/kaggle/working/last_rcan.pth'
METRICS_PATH    = '/kaggle/working/best_metrics.json'

best_psnr  = -float('inf')
es_counter = 0
train_history = []

print(f'Training up to {epochs} epochs | batch={BATCH_SIZE} | {device}')
print(f'Early stopping patience: {ES_PATIENCE} epochs')
print('-' * 70)

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Ep {epoch:03d}/{epochs}', leave=False)

    for lrp, hrp in pbar:
        lrp = lrp.to(device); hrp = hrp.to(device)
        optimizer.zero_grad()
        pred = model(lrp)
        _, _, h, w = hrp.shape
        pred = pred[:,:,:h,:w]
        loss = criterion(pred, hrp)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.5f}')

    avg_loss = running_loss / len(train_loader)
    scheduler.step()
    lr_now   = optimizer.param_groups[0]['lr']
    val_psnr = validate(model, val_loader, device)

    if val_psnr > best_psnr:
        best_psnr  = val_psnr
        es_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        with open(METRICS_PATH, 'w') as mf:
            _json.dump({'epoch':epoch,'val_psnr':val_psnr}, mf)
        tag = '  [saved]'
    else:
        es_counter += 1
        tag = f'  [no improve {es_counter}/{ES_PATIENCE}]'

    print(f'Ep {epoch:03d}/{epochs} | loss:{avg_loss:.5f} | '
          f'PSNR:{val_psnr:.2f}dB | Best:{best_psnr:.2f}dB | '
          f'lr:{lr_now:.1e}{tag}')
    train_history.append(dict(epoch=epoch, loss=avg_loss, psnr=val_psnr))

    if es_counter >= ES_PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

torch.save(model.state_dict(), LAST_PATH)
print('-' * 70)
with open(METRICS_PATH) as mf: bm = _json.load(mf)
print(f'Best — Ep:{bm["epoch"]} PSNR:{bm["val_psnr"]:.2f}dB')


Training up to 200 epochs | batch=4 | cuda
Early stopping patience: 25 epochs
----------------------------------------------------------------------


Ep 001/200 | loss:0.15534 | PSNR:26.23dB | Best:26.23dB | lr:2.8e-05  [saved]


Ep 002/200 | loss:0.11614 | PSNR:26.73dB | Best:26.73dB | lr:4.6e-05  [saved]


Ep 003/200 | loss:0.10823 | PSNR:27.08dB | Best:27.08dB | lr:6.4e-05  [saved]


Ep 004/200 | loss:0.10400 | PSNR:27.05dB | Best:27.08dB | lr:8.2e-05  [no improve 1/25]


...

Ep 171/200 | loss:0.08077 | PSNR:28.69dB | Best:28.73dB | lr:6.3e-06  [no improve 21/25]


Ep 172/200 | loss:0.08013 | PSNR:28.66dB | Best:28.73dB | lr:6.0e-06  [no improve 22/25]


Ep 173/200 | loss:0.08057 | PSNR:28.65dB | Best:28.73dB | lr:5.6e-06  [no improve 23/25]


Ep 174/200 | loss:0.08062 | PSNR:28.68dB | Best:28.73dB | lr:5.3e-06  [no improve 24/25]


Ep 175/200 | loss:0.08030 | PSNR:28.67dB | Best:28.73dB | lr:5.0e-06  [no improve 25/25]
Early stopping at epoch 175.
----------------------------------------------------------------------
Best — Ep:150 PSNR:28.73dB


## Section 9 — Inference with x8 TTA

In [9]:
assert os.path.exists(CHECKPOINT_PATH), 'No checkpoint. Run Section 8 first.'
assert os.path.exists(METRICS_PATH),    'No metrics. Run Section 8 first.'

with open(METRICS_PATH) as mf: bm = _json.load(mf)
print(f'Checkpoint: Ep {bm["epoch"]} | PSNR:{bm["val_psnr"]:.2f}dB')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_infer = RCAN(
    scale=SCALE, n_resgroups=5, n_resblocks=10,
    n_feats=64, reduction=16, in_chans=1
).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
sd   = ckpt['state_dict'] if isinstance(ckpt,dict) and 'state_dict' in ckpt else ckpt
model_infer.load_state_dict(sd)
model_infer.eval()

w_mean = next(model_infer.parameters()).abs().mean().item()
print(f'Weight abs mean: {w_mean:.6f}')
assert w_mean > 1e-4, 'Weights look like random init.'
print('Weights OK.')


def _tta_fwd(x, i):
    if i >= 4: x = torch.flip(x, [-1])
    k = i % 4
    if k: x = torch.rot90(x, k=k, dims=[-2,-1])
    return x

def _tta_inv(x, i):
    k = i % 4
    if k: x = torch.rot90(x, k=(4-k)%4, dims=[-2,-1])
    if i >= 4: x = torch.flip(x, [-1])
    return x


TEST_LR_DIR    = '/kaggle/input/datasets/kesavas777/semiconductors/KLA AI - HACK/test/NoisyLR'
SUBMISSION_DIR = '/kaggle/working/submission'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

files = sorted(os.listdir(TEST_LR_DIR))
assert len(files) > 0, f'No test files in {TEST_LR_DIR}'
print(f'Inference on {len(files)} test images...')

with torch.no_grad():
    for f in tqdm(files, desc='x8 TTA'):
        img = np.load(os.path.join(TEST_LR_DIR, f))
        if img.ndim == 2: img = img[np.newaxis]
        inp = torch.from_numpy(img.copy()).float().unsqueeze(0).to(device)
        preds = []
        for i in range(8):
            a = _tta_fwd(inp, i)
            p = model_infer(a)
            _, _, hi, wi = a.shape
            p = p[:,:,:hi*SCALE,:wi*SCALE]
            preds.append(_tta_inv(p, i))
        out    = torch.stack(preds).mean(0).clamp(0, 1)
        out_np = out.squeeze().cpu().numpy().astype(np.float32)
        assert out_np.shape == (256, 256), f'{f}: got {out_np.shape}'
        np.save(os.path.join(SUBMISSION_DIR, f), out_np)

sample = np.load(os.path.join(SUBMISSION_DIR, files[0]))
print(f'Sample: shape={sample.shape} dtype={sample.dtype} '
      f'min={sample.min():.4f} max={sample.max():.4f}')
print(f'Saved {len(files)} predictions.')


Checkpoint: Ep 150 | PSNR:28.73dB
Weight abs mean: 0.164939
Weights OK.
Inference on 200 test images...


x8 TTA: 100%|██████████| 200/200 [01:01<00:00,  3.25it/s]

Sample: shape=(256, 256) dtype=float32 min=0.0031 max=1.0000
Saved 200 predictions.


## Section 10 — Submission CSV (verbatim competition template)

In [10]:
submission_dir = '/kaggle/working/submission'

rows = []

files = sorted([f for f in os.listdir(submission_dir) if f.endswith('.npy')])

for idx, file in enumerate(files, start=1):

    path = os.path.join(submission_dir, file)

    # load numpy array
    arr = np.load(path)

    # convert array to bytes
    buffer = BytesIO()
    np.save(buffer, arr)

    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({
        'id': idx,
        'npy_base64': encoded
    })

df = pd.DataFrame(rows)

df.to_csv('/kaggle/working/submission.csv', index=False)

print('Submission created with', len(df), 'rows')


Submission created with 200 rows
